In [ ]:
from scipy.io import loadmat
import numpy as np
from sklearn.preprocessing import StandardScaler
import pandas as pd
import gc

In [ ]:
# Load all 3 subjects
sub1 = loadmat("sub1_comp.mat")
sub2 = loadmat("sub2_comp.mat")
sub3 = loadmat("sub3_comp.mat")

print("Sub1 train:", sub1['train_data'].shape, "-> 62 channels")
print("Sub2 train:", sub2['train_data'].shape, "-> 48 channels")
print("Sub3 train:", sub3['train_data'].shape, "-> 64 channels")

Sub1 train: (400000, 62) -> 62 channels
Sub2 train: (400000, 48) -> 48 channels
Sub3 train: (400000, 64) -> 64 channels


## Smart Channel Removal

We identify **actually unused/bad channels** using two criteria:

1. **Variance filter** — Remove dead/flat electrodes (near-zero variance) and noisy electrodes (extremely high variance)
2. **Correlation filter** — Remove channels that have zero or near-zero correlation with any of the 5 finger flexion signals

We apply both filters to each subject independently, then unify by keeping only the first N channels that survived in ALL subjects.

In [ ]:
def find_good_channels(train_data, train_dg, subject_name,
                       var_low_percentile=1, var_high_percentile=99,
                       corr_threshold=0.01):
    """
    Identify good channels using variance and correlation filters.
    """
    n_channels = train_data.shape[1]

    # --- FILTER 1: Variance ---
    variances = np.var(train_data, axis=0)
    var_low = np.percentile(variances, var_low_percentile)
    var_high = np.percentile(variances, var_high_percentile)

    var_good = set()
    var_bad_low = []
    var_bad_high = []
    for ch in range(n_channels):
        if variances[ch] < var_low:
            var_bad_low.append(ch)
        elif variances[ch] > var_high:
            var_bad_high.append(ch)
        else:
            var_good.add(ch)

    print(f"\n{subject_name} - Variance Filter:")
    print(f"  Dead/flat channels (low var):  {var_bad_low}")
    print(f"  Noisy channels (high var):     {var_bad_high}")
    print(f"  Channels passing variance:     {len(var_good)}/{n_channels}")

    # --- FILTER 2: Correlation with finger flexion ---
    corr_good = set()
    corr_bad = []

    for ch in range(n_channels):
        ch_corrs = []
        for f in range(train_dg.shape[1]):
            r = np.abs(np.corrcoef(train_data[:, ch], train_dg[:, f])[0, 1])
            ch_corrs.append(r)
        max_corr = max(ch_corrs)

        if max_corr >= corr_threshold:
            corr_good.add(ch)
        else:
            corr_bad.append(ch)

    print(f"\n{subject_name} - Correlation Filter:")
    print(f"  Channels with no finger correlation: {corr_bad}")
    print(f"  Channels passing correlation:        {len(corr_good)}/{n_channels}")

    # --- COMBINE: Must pass BOTH filters ---
    good_channels = sorted(var_good & corr_good)
    removed = sorted(set(range(n_channels)) - set(good_channels))

    print(f"\n{subject_name} - Combined Result:")
    print(f"  Good channels: {len(good_channels)}/{n_channels}")
    print(f"  Removed channels: {removed}")

    return good_channels

In [ ]:
# Apply filters to each subject
good_ch_1 = find_good_channels(sub1['train_data'], sub1['train_dg'], "Subject 1")
good_ch_2 = find_good_channels(sub2['train_data'], sub2['train_dg'], "Subject 2")
good_ch_3 = find_good_channels(sub3['train_data'], sub3['train_dg'], "Subject 3")


Subject 1 - Variance Filter:
  Dead/flat channels (low var):  [23]
  Noisy channels (high var):     [54]
  Channels passing variance:     60/62

Subject 1 - Correlation Filter:
  Channels with no finger correlation: [12]
  Channels passing correlation:        61/62

Subject 1 - Combined Result:
  Good channels: 59/62
  Removed channels: [12, 23, 54]

Subject 2 - Variance Filter:
  Dead/flat channels (low var):  [47]
  Noisy channels (high var):     [20]
  Channels passing variance:     46/48

Subject 2 - Correlation Filter:
  Channels with no finger correlation: [16]
  Channels passing correlation:        47/48

Subject 2 - Combined Result:
  Good channels: 45/48
  Removed channels: [16, 20, 47]

Subject 3 - Variance Filter:
  Dead/flat channels (low var):  [42]
  Noisy channels (high var):     [57]
  Channels passing variance:     62/64

Subject 3 - Correlation Filter:
  Channels with no finger correlation: []
  Channels passing correlation:        64/64

Subject 3 - Combined Result:

In [ ]:
# Unify: keep the minimum number of good channels across all subjects
N_UNIFIED = min(len(good_ch_1), len(good_ch_2), len(good_ch_3))

print(f"Good channels per subject:")
print(f"  Subject 1: {len(good_ch_1)} out of 62")
print(f"  Subject 2: {len(good_ch_2)} out of 48")
print(f"  Subject 3: {len(good_ch_3)} out of 64")
print(f"\nUnified channel count: {N_UNIFIED}")

final_ch_1 = good_ch_1[:N_UNIFIED]
final_ch_2 = good_ch_2[:N_UNIFIED]
final_ch_3 = good_ch_3[:N_UNIFIED]

print(f"\nFinal channel indices used:")
print(f"  Subject 1: {final_ch_1}")
print(f"  Subject 2: {final_ch_2}")
print(f"  Subject 3: {final_ch_3}")

Good channels per subject:
  Subject 1: 59 out of 62
  Subject 2: 45 out of 48
  Subject 3: 62 out of 64

Unified channel count: 45

Final channel indices used:
  Subject 1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
  Subject 2: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
  Subject 3: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 45]


In [ ]:
# Apply channel selection
sub1_train = sub1['train_data'][:, final_ch_1]
sub1_test  = sub1['test_data'][:, final_ch_1]
sub1_dg    = sub1['train_dg']

sub2_train = sub2['train_data'][:, final_ch_2]
sub2_test  = sub2['test_data'][:, final_ch_2]
sub2_dg    = sub2['train_dg']

sub3_train = sub3['train_data'][:, final_ch_3]
sub3_test  = sub3['test_data'][:, final_ch_3]
sub3_dg    = sub3['train_dg']

print(f"After smart channel removal ({N_UNIFIED} channels each):")
print(f"  Sub1 train: {sub1_train.shape}")
print(f"  Sub2 train: {sub2_train.shape}")
print(f"  Sub3 train: {sub3_train.shape}")

After smart channel removal (45 channels each):
  Sub1 train: (400000, 45)
  Sub2 train: (400000, 45)
  Sub3 train: (400000, 45)


## Downsample Before Merging

The original data is sampled at **1000 Hz**, which is much higher than needed for finger flexion prediction (the finger data itself is only 25 Hz, upsampled to 1000 Hz).

We downsample by a factor of 4 (to 250 Hz), which:
- Cuts memory usage by 4x
- Still preserves signals up to 125 Hz (well above the useful motor cortex frequency range)
- Makes all subsequent steps 4x faster

In [ ]:
# Downsample by factor of 4: 1000 Hz -> 250 Hz
DOWNSAMPLE_FACTOR = 4

sub1_train_ds = sub1_train[::DOWNSAMPLE_FACTOR]
sub1_dg_ds    = sub1_dg[::DOWNSAMPLE_FACTOR]

sub2_train_ds = sub2_train[::DOWNSAMPLE_FACTOR]
sub2_dg_ds    = sub2_dg[::DOWNSAMPLE_FACTOR]

sub3_train_ds = sub3_train[::DOWNSAMPLE_FACTOR]
sub3_dg_ds    = sub3_dg[::DOWNSAMPLE_FACTOR]

print(f"After downsampling (factor={DOWNSAMPLE_FACTOR}):")
print(f"  Sub1: {sub1_train.shape} -> {sub1_train_ds.shape}")
print(f"  Sub2: {sub2_train.shape} -> {sub2_train_ds.shape}")
print(f"  Sub3: {sub3_train.shape} -> {sub3_train_ds.shape}")

# Free original full-size arrays from memory
del sub1, sub2, sub3
del sub1_train, sub2_train, sub3_train
del sub1_dg, sub2_dg, sub3_dg
gc.collect()
print("\nFreed original data from memory.")

After downsampling (factor=4):
  Sub1: (400000, 45) -> (100000, 45)
  Sub2: (400000, 45) -> (100000, 45)
  Sub3: (400000, 45) -> (100000, 45)

Freed original data from memory.


In [ ]:
# Merge all 3 subjects
X_train_all = np.concatenate([sub1_train_ds, sub2_train_ds, sub3_train_ds], axis=0)
y_train_all = np.concatenate([sub1_dg_ds, sub2_dg_ds, sub3_dg_ds], axis=0)

# Free individual subject arrays
del sub1_train_ds, sub2_train_ds, sub3_train_ds
del sub1_dg_ds, sub2_dg_ds, sub3_dg_ds
gc.collect()

print(f"Merged train data: {X_train_all.shape}")
print(f"Merged train_dg:   {y_train_all.shape}")
print(f"\nMemory: ~{X_train_all.nbytes / 1e9:.2f} GB (was ~{X_train_all.shape[0] * DOWNSAMPLE_FACTOR * X_train_all.shape[1] * 8 / 1e9:.2f} GB before downsampling)")

Merged train data: (300000, 45)
Merged train_dg:   (300000, 5)

Memory: ~0.05 GB (was ~0.43 GB before downsampling)


## Batch Windowing + Feature Extraction

Instead of creating the entire windowed array in memory (which would be huge), we:
1. Process the data in small chunks
2. Extract features from each chunk immediately
3. Only keep the extracted features (much smaller than the windowed data)

This way, the giant (samples × window × channels) array never exists in memory all at once.

**Window size is also adjusted:** Original was 100 at 1000 Hz = 100ms window. At 250 Hz, we use 25 to keep the same 100ms window.

In [ ]:
def windowed_feature_extraction_batched(X, y, window_size=25, batch_size=50000):
    """
    Combined windowing + feature extraction in batches.
    Never creates the full windowed array in memory.

    For each window, extracts: mean, std, min, max per channel.
    """
    n_samples = len(X) - window_size
    n_channels = X.shape[1]
    n_features = n_channels * 4  # mean, std, min, max

    all_features = []
    all_labels = []

    # Process in batches
    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        batch_features = []

        for i in range(start, end):
            window = X[i:i + window_size]  # shape: (window_size, n_channels)

            mean_f = np.mean(window, axis=0)
            std_f  = np.std(window, axis=0)
            min_f  = np.min(window, axis=0)
            max_f  = np.max(window, axis=0)

            features = np.concatenate([mean_f, std_f, min_f, max_f])
            batch_features.append(features)

        all_features.append(np.array(batch_features))
        all_labels.append(y[window_size + start : window_size + end])

        processed = end
        print(f"  Processed {processed:,}/{n_samples:,} samples "
              f"({100*processed/n_samples:.1f}%)", end='\r')

    print("  Done!" + " " * 50)
    return np.concatenate(all_features, axis=0), np.concatenate(all_labels, axis=0)


# Window size adjusted for downsampled rate:
# Original: 100 samples at 1000 Hz = 100ms
# Now:      25 samples at 250 Hz  = 100ms (same time window)
WINDOW_SIZE = 25

print(f"Running batched windowing + feature extraction...")
print(f"  Window size: {WINDOW_SIZE} samples ({WINDOW_SIZE * DOWNSAMPLE_FACTOR}ms)")
print(f"  Input shape: {X_train_all.shape}")

X_feature, y_feature = windowed_feature_extraction_batched(
    X_train_all, y_train_all, window_size=WINDOW_SIZE, batch_size=50000
)

print(f"\nOutput:")
print(f"  X_feature: {X_feature.shape}")
print(f"  y_feature: {y_feature.shape}")
print(f"  Memory: ~{X_feature.nbytes / 1e6:.1f} MB")

Running batched windowing + feature extraction...
  Window size: 25 samples (100ms)
  Input shape: (300000, 45)
  Done!                                                  

Output:
  X_feature: (299975, 180)
  y_feature: (299975, 5)
  Memory: ~432.0 MB


In [ ]:
# Free the raw merged data — we only need features from here
del X_train_all, y_train_all
gc.collect()
print("Freed raw data from memory.")

Freed raw data from memory.


In [ ]:
# Scaling the Features
Scaler = StandardScaler()
X_scaled = Scaler.fit_transform(X_feature)

print(f"X_scaled: {X_scaled.shape}")
print(f"y_feature: {y_feature.shape}")

X_scaled: (299975, 180)
y_feature: (299975, 5)


In [ ]:
# Create feature names
channel_names = [f"ch_{i}" for i in range(N_UNIFIED)]

feature_names = []
for ch in channel_names:
    feature_names.append(f"{ch}_mean")
    feature_names.append(f"{ch}_std")
    feature_names.append(f"{ch}_min")
    feature_names.append(f"{ch}_max")

print(f"Total features: {len(feature_names)}")
print(f"First 10: {feature_names[:10]}")

Total features: 180
First 10: ['ch_0_mean', 'ch_0_std', 'ch_0_min', 'ch_0_max', 'ch_1_mean', 'ch_1_std', 'ch_1_min', 'ch_1_max', 'ch_2_mean', 'ch_2_std']


In [ ]:
# Convert to DataFrame
data_df = pd.DataFrame(X_scaled, columns=feature_names)

print(data_df.head())

   ch_0_mean  ch_0_std  ch_0_min  ch_0_max  ch_1_mean  ch_1_std  ch_1_min  \
0  -0.514795  1.091586 -1.274512  0.705400  -0.179217  0.606125  1.334093   
1  -0.496313  1.118889 -1.201559  0.601355  -0.207266  0.568789  1.335528   
2  -0.470624  1.129660 -1.155380  0.494897  -0.251235  0.536238  1.298337   
3  -0.446978  1.116859 -1.152298  0.384238  -0.282782  0.502985  1.220270   
4  -0.407342  1.083696 -1.177914  0.263582  -0.319440  0.467110  1.096205   

   ch_1_max  ch_2_mean  ch_2_std  ...  ch_42_min  ch_42_max  ch_43_mean  \
0  0.227621  -0.145960 -0.230512  ...   0.000316   -0.47275     0.97801   
1  0.234725  -0.203138 -0.281917  ...   0.000316   -0.47275     0.97801   
2  0.238376  -0.263056 -0.323483  ...   0.000316   -0.47275     0.97801   
3  0.266506  -0.326344 -0.356999  ...   0.000316   -0.47275     0.97801   
4  0.282169  -0.389247 -0.393966  ...   0.000316   -0.47275     0.97801   

   ch_43_std  ch_43_min  ch_43_max  ch_44_mean  ch_44_std  ch_44_min  \
0  -0.373298  